# Forschungsfrage 2 - Fehlende Werte: Traditioneller Ansatz (Median-/Modus-Imputation)

Klassische statistische Imputation ohne jegliches Lernmodell:

- **Numerisches Zielattribut `price`:** Imputation durch den **Median** von `price`,
  gruppiert nach der (falls vorhandenen) `parent_category` - mit Rückfall auf den
  globalen Median, wenn `parent_category` selbst fehlt oder in der Trainingsmenge
  nicht vorkam.
- **Kategoriales Zielattribut `parent_category`:** Imputation durch den **Modus**
  (häufigste Kategorie) von `parent_category`, gruppiert nach `thread_category` -
  mit Rückfall auf den globalen Modus.

Alle Gruppenstatistiken werden ausschließlich auf dem `train`-Split berechnet,
die Bewertung erfolgt ausschließlich auf dem `test`-Split (identisch zu den
anderen beiden TF2-Notebooks, um Vergleichbarkeit zu gewährleisten).


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import time
import os

from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score

os.makedirs("results", exist_ok=True)


## 2. Numerisches Zielattribut: `price` (Median je `parent_category`)

In [2]:
price_df = pd.read_csv("benchmark/tf2_missing_price.csv")
price_truth = pd.read_csv("benchmark/tf2_missing_price_groundtruth.csv").set_index("row_id")["price_true"]

train_price = price_df[price_df["split"] == "train"]
global_median_price = train_price["price_input"].median()
group_median_price = train_price.groupby("parent_category")["price_input"].median()

print(f"Globaler Median (train): {global_median_price:.2f} USD")
print(f"Anzahl parent_category-Gruppen mit eigenem Median: {group_median_price.notna().sum()}")

def impute_price(row):
    pc = row["parent_category"]
    if pd.notna(pc) and pc in group_median_price.index and pd.notna(group_median_price[pc]):
        return group_median_price[pc]
    return global_median_price

t0 = time.time()
test_price = price_df[price_df["split"] == "test"].copy()
test_price["price_pred_median"] = test_price.apply(impute_price, axis=1)
train_time_price = time.time() - t0  # Nachschlagen einer Lookup-Tabelle - keine echte "Trainingszeit"

test_ids = test_price["row_id"].values
y_true_price = price_truth.loc[test_ids].values
y_pred_price = test_price["price_pred_median"].values

mae = mean_absolute_error(y_true_price, y_pred_price)
rmse = np.sqrt(mean_squared_error(y_true_price, y_pred_price))
n_fallback = (~test_price["parent_category"].isin(group_median_price.dropna().index)).sum()
print(f"price - MAE: {mae:.3f}  RMSE: {rmse:.3f}  (n_test={len(test_ids)}, davon {n_fallback} mit globalem Rückfall-Median)")

price_results = pd.DataFrame({"row_id": test_ids, "price_true": y_true_price, "price_pred_median": y_pred_price})
price_results.to_csv("results/tf2_medianmode_price_predictions.csv", index=False)


Globaler Median (train): 50.30 USD
Anzahl parent_category-Gruppen mit eigenem Median: 12
price - MAE: 189.401  RMSE: 362.302  (n_test=164, davon 57 mit globalem Rückfall-Median)


## 3. Kategoriales Zielattribut: `parent_category` (Modus je `thread_category`)

In [3]:
cat_df = pd.read_csv("benchmark/tf2_missing_parent_category.csv")
cat_truth = pd.read_csv("benchmark/tf2_missing_parent_category_groundtruth.csv").set_index("row_id")["parent_category_true"]

train_cat = cat_df[cat_df["split"] == "train"]
global_mode_cat = train_cat["parent_category_input"].mode().iloc[0]
group_mode_cat = train_cat.groupby("thread_category")["parent_category_input"].agg(
    lambda s: s.mode().iloc[0] if len(s.mode()) > 0 else global_mode_cat
)

print(f"Globaler Modus (train): {global_mode_cat}")
print(f"Anzahl thread_category-Gruppen mit eigenem Modus: {len(group_mode_cat)}")

def impute_category(row):
    tc = row["thread_category"]
    if pd.notna(tc) and tc in group_mode_cat.index:
        return group_mode_cat[tc]
    return global_mode_cat

t0 = time.time()
test_cat = cat_df[cat_df["split"] == "test"].copy()
test_cat["parent_category_pred_median"] = test_cat.apply(impute_category, axis=1)
train_time_cat = time.time() - t0

test_ids2 = test_cat["row_id"].values
y_true_cat = cat_truth.loc[test_ids2].values
y_pred_cat = test_cat["parent_category_pred_median"].values

acc = accuracy_score(y_true_cat, y_pred_cat)
macro_f1 = f1_score(y_true_cat, y_pred_cat, average="macro")
n_fallback2 = (~test_cat["thread_category"].isin(group_mode_cat.index)).sum()
print(f"parent_category - Accuracy: {acc:.3f}  Macro-F1: {macro_f1:.3f}  (n_test={len(test_ids2)}, davon {n_fallback2} mit globalem Rückfall-Modus)")

cat_results = pd.DataFrame({"row_id": test_ids2, "parent_category_true": y_true_cat, "parent_category_pred_median": y_pred_cat})
cat_results.to_csv("results/tf2_medianmode_parent_category_predictions.csv", index=False)


Globaler Modus (train): Computers & Electronics
Anzahl thread_category-Gruppen mit eigenem Modus: 42
parent_category - Accuracy: 0.912  Macro-F1: 0.858  (n_test=125, davon 0 mit globalem Rückfall-Modus)


## 4. Metriken und Laufzeit-Log speichern

In [4]:
metrics = {
    "experiment": "TF2_FehlendeWerte", "method": "Median_Modus",
    "price_mae": mae, "price_rmse": rmse, "price_n_test": len(test_ids),
    "parent_category_accuracy": acc, "parent_category_macro_f1": macro_f1, "parent_category_n_test": len(test_ids2),
    "train_time_price_sec": train_time_price, "train_time_parent_category_sec": train_time_cat,
}
with open("results/tf2_medianmode_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([
    {"experiment": "TF2_FehlendeWerte_price", "method": "Median_Modus", "n_items": len(test_ids),
     "wall_time_sec": train_time_price, "input_tokens": 0, "output_tokens": 0,
     "estimated_cost_usd": 0.0, "model_name": "median_je_parent_category (kein Modell, kein LLM/API)"},
    {"experiment": "TF2_FehlendeWerte_parent_category", "method": "Median_Modus", "n_items": len(test_ids2),
     "wall_time_sec": train_time_cat, "input_tokens": 0, "output_tokens": 0,
     "estimated_cost_usd": 0.0, "model_name": "modus_je_thread_category (kein Modell, kein LLM/API)"},
])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf2_medianmode_price_predictions.csv, results/tf2_medianmode_parent_category_predictions.csv, results/tf2_medianmode_metrics.json")


Gespeichert: results/tf2_medianmode_price_predictions.csv, results/tf2_medianmode_parent_category_predictions.csv, results/tf2_medianmode_metrics.json
